# Notebook 10b -- Quantum Counting and Amplitude Estimation

**Prerequisites:** Notebooks 08-09. Familiarity with QPE and Grover's algorithm.

In the previous notebook we saw how Shor's algorithm uses Quantum Phase
Estimation (QPE) to find the period of modular exponentiation, enabling
efficient integer factoring. In this notebook we explore two more powerful
applications of QPE: **quantum counting** and **amplitude estimation**.

Quantum counting tells us *how many* solutions a search problem has, while
amplitude estimation generalizes this to estimate the probability amplitude
of any "good" subspace. Both provide quadratic speedups over their classical
counterparts and serve as core subroutines in quantum algorithms for Monte
Carlo simulation, optimization, and machine learning.

By the end of this notebook you will be able to:

1. **Explain** how quantum counting uses QPE on the Grover iterate to
   estimate the number of solutions to a search problem.
2. **Run** quantum counting on a small oracle and interpret the results.
3. **Describe** how amplitude estimation generalizes quantum counting.
4. **Compare** standard and iterative amplitude estimation in terms of
   accuracy and resource usage.

In [1]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/ampest"
	"github.com/splch/goqu/algorithm/counting"
	"github.com/splch/goqu/algorithm/grover"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/viz"
)

## Quantum Counting

**Quantum counting** answers the question: *how many solutions does a search
problem have?* This is useful when we need to know the number of marked states
$M$ in a search space of size $N = 2^n$ before running Grover's search (since
the optimal iteration count depends on $M$).

The algorithm works by applying **QPE to the Grover iterate** $Q$. The Grover
operator has eigenvalues $e^{\pm 2i\theta}$ where $\sin^2(\theta) = M/N$.
QPE estimates $\theta$, from which we recover:

$$M = N \sin^2(\theta)$$

With $t$ phase bits, the estimate achieves $t$ bits of precision. This gives a
quadratic speedup over classical counting, which requires $O(N)$ oracle queries.

In [2]:
%%
ctx := context.Background()

// Use quantum counting to estimate the number of solutions
// to a Grover oracle. We mark states |01> and |10> (2 solutions
// out of 4 total states in a 2-qubit search space).
oracle := grover.PhaseOracle([]int{0b01, 0b10}, 2)

result, err := counting.Run(ctx, counting.Config{
	NumQubits:    2,
	Oracle:       oracle,
	NumPhaseBits: 4,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("Estimated count: %.1f (expected: 2)\n", result.Count)
fmt.Printf("Phase: %.4f\n", result.Phase)
fmt.Println("Measurement counts:", result.Counts)

svg := viz.Histogram(result.Counts, viz.WithTitle("Quantum Counting: 2 Solutions in 4 States"))
gonbui.DisplayHTML(svg)

Estimated count: 2.0 (expected: 2)
Phase: 0.7500
Measurement counts: map[000010:116 000011:123 010010:112 010011:127 100010:134 100011:128 110010:123 110011:137]


Quantum Counting: 2 Solutions in 4 States 
 
 
 
 0 
 
 20 
 
 40 
 
 60 
 
 80 
 
 100 
 
 120 
 
 140 
 
 000010 
 
 000011 
 
 010010 
 
 010011 
 
 100010 
 
 100011 
 
 110010 
 
 110011

The histogram shows peaks at phase register values whose binary fractions
approximate $\theta / \pi$. From the dominant peak we extract $\theta$ and
compute $M = N \sin^2(\theta) \approx 2$, confirming that our 2-qubit search
space has exactly 2 marked states.

## Amplitude Estimation

**Amplitude estimation** generalizes quantum counting. Instead of counting
the number of solutions, it estimates the **probability amplitude** $a$ of a
"good" subspace prepared by a state-preparation circuit $A$.

Given:
- A state-preparation circuit $A$ that creates $A|0\rangle = \sqrt{1-a^2}|\psi_0\rangle + a|\psi_1\rangle$
- An oracle $S_f$ that marks the "good" states $|\psi_1\rangle$

Standard amplitude estimation builds the **Grover iterate**
$Q = A \cdot S_0 \cdot A^\dagger \cdot S_f$ and applies QPE to estimate the
phase $\theta$ where $a = \sin(\pi\theta)$.

This is a core subroutine in:
- **Monte Carlo methods** (quadratic speedup for financial derivative pricing)
- **Quantum machine learning** (estimating inner products)
- **Optimization** (evaluating objective functions)

In [3]:
%%
ctx := context.Background()

// Amplitude estimation: estimate the amplitude of |11> in the
// uniform superposition H|0>H|0> = (|00>+|01>+|10>+|11>)/2.
// The amplitude of |11> is 1/2, so probability = 1/4.
statePrep, err := builder.New("prep", 2).H(0).H(1).Build()
if err != nil {
	panic(err)
}

oracle := grover.PhaseOracle([]int{0b11}, 2)

result, err := ampest.Run(ctx, ampest.Config{
	StatePrep:    statePrep,
	Oracle:       oracle,
	NumQubits:    2,
	NumPhaseBits: 4,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("Amplitude: %.4f (expected: 0.5000)\n", result.Amplitude)
fmt.Printf("Probability: %.4f (expected: 0.2500)\n", result.Probability)
fmt.Printf("Phase: %.4f\n", result.Phase)
fmt.Println("Measurement counts:", result.Counts)

svg := viz.Histogram(result.Counts, viz.WithTitle("Amplitude Estimation: P(|11>) in Uniform State"))
gonbui.DisplayHTML(svg)

Amplitude: 0.5556 (expected: 0.5000)
Probability: 0.3087 (expected: 0.2500)
Phase: 0.3125
Measurement counts: map[000001:3 000010:5 000011:6 000100:2 000101:7 000110:11 001000:1 001001:2 001010:70 001100:1 001101:51 001110:1 010000:1 010010:5 010011:5 010100:1 010101:19 010110:16 010111:3 011001:2 011010:51 011011:1 011100:2 011101:67 011110:3 011111:4 100000:1 100001:4 100010:4 100100:1 100101:13 100110:11 100111:2 101001:1 101010:65 101011:3 101100:2 101101:52 101110:3 101111:2 110000:1 110001:9 110010:8 110011:5 110100:1 110101:53 110110:53 111000:1 111001:15 111010:172 111011:2 111100:1 111101:158 111110:15 111111:2]


Amplitude Estimation: P(|11>) in Uniform State 
 
 
 
 0 
 
 50 
 
 100 
 
 150 
 
 200 
 
 000001 
 
 000010 
 
 000011 
 
 000100 
 
 000101 
 
 000110 
 
 001000 
 
 001001 
 
 001010 
 
 001100 
 
 001101 
 
 001110 
 
 010000 
 
 010010 
 
 010011 
 
 010100 
 
 010101 
 
 010110 
 
 010111 
 
 011001 
 
 011010 
 
 011011 
 
 011100 
 
 011101 
 
 011110 
 
 011111 
 
 100000 
 
 100001 
 
 100010 
 
 100100 
 
 100101 
 
 100110 
 
 100111 
 
 101001 
 
 101010 
 
 101011 
 
 101100 
 
 101101 
 
 101110 
 
 101111 
 
 110000 
 
 110001 
 
 110010 
 
 110011 
 
 110100 
 
 110101 
 
 110110 
 
 111000 
 
 111001 
 
 111010 
 
 111011 
 
 111100 
 
 111101 
 
 111110 
 
 111111

## Predict, Then Verify

**Question:** A 3-qubit search space has $N = 8$ states. If we mark states
$|001\rangle$, $|011\rangle$, and $|101\rangle$ (3 solutions), what count
should quantum counting return?

**Prediction:** The search space has $N = 8$ states with $M = 3$ solutions.
Quantum counting estimates $M$ via $M = N \sin^2(\theta)$. With enough phase
bits the estimate should be close to $3$.

Let's verify.

In [4]:
%%
ctx := context.Background()

// 3-qubit search space: mark |001>=1, |011>=3, |101>=5
oracle := grover.PhaseOracle([]int{0b001, 0b011, 0b101}, 3)

result, err := counting.Run(ctx, counting.Config{
	NumQubits:    3,
	Oracle:       oracle,
	NumPhaseBits: 6,
	Shots:        1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("Estimated count: %.2f (expected: 3)\n", result.Count)
fmt.Printf("Phase: %.4f\n", result.Phase)
fmt.Printf("\nPrediction confirmed: quantum counting correctly estimates M ≈ 3.\n")

Estimated count: 2.84 (expected: 3)
Phase: 0.7031

Prediction confirmed: quantum counting correctly estimates M ≈ 3.


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Why does quantum counting apply QPE to the Grover iterate rather than to the oracle directly?
2. How does the number of phase bits affect the precision of the count estimate?
3. What is the main resource advantage of iterative amplitude estimation over standard amplitude estimation?

---

## Exercises

### Exercise 1 -- Quantum Counting with a Single Solution

Use quantum counting to estimate the number of solutions when only one state
is marked in a 3-qubit search space ($N = 8$, $M = 1$). How close is the
estimate to $1$? Try increasing `NumPhaseBits` to see how precision improves.

In [5]:
%%
// Exercise 1: Quantum counting with a single marked state
// Search space: 3 qubits (N=8), mark only |101> (M=1)
//
// TODO: Create a phase oracle marking a single state
// oracle := grover.PhaseOracle([]int{???}, 3)
//
// TODO: Run quantum counting with 4 phase bits
// result, err := counting.Run(ctx, counting.Config{
//     NumQubits:    ???,
//     Oracle:       oracle,
//     NumPhaseBits: ???,
//     Shots:        1000,
// })
//
// TODO: Print the estimated count and compare to expected M=1
//
// TODO: Try again with NumPhaseBits=6 and NumPhaseBits=8.
//       How does precision change?
_ = context.Background()
fmt.Println("Uncomment the code above and fill in the config!")

Uncomment the code above and fill in the config!


### Exercise 2 -- Compare Standard vs Iterative Amplitude Estimation

Standard amplitude estimation uses QPE with a full ancilla register, building
an $O(2^t)$-depth circuit. **Iterative amplitude estimation** avoids the large
ancilla register by using a single ancilla qubit and progressively doubling the
Grover power, achieving similar precision with lower circuit depth.

Run both methods on the same problem (estimating the amplitude of $|11\rangle$
in the uniform 2-qubit superposition) and compare their accuracy.

In [6]:
%%
// Exercise 2: Standard vs Iterative Amplitude Estimation
// Estimate the amplitude of |11> in the uniform 2-qubit superposition.
// The amplitude of |11> is 1/2, so probability = 1/4.
//
// TODO: Build the state preparation circuit (H on both qubits)
// statePrep, err := builder.New("prep", 2).H(0).H(1).Build()
//
// TODO: Create a phase oracle marking |11>
// oracle := grover.PhaseOracle([]int{???}, 2)
//
// TODO: Run standard amplitude estimation with 4 phase bits
// stdResult, err := ampest.Run(ctx, ampest.Config{
//     StatePrep:    statePrep,
//     Oracle:       oracle,
//     NumQubits:    ???,
//     NumPhaseBits: ???,
//     Shots:        1000,
// })
//
// TODO: Run iterative amplitude estimation with 4 max iterations
// iterResult, err := ampest.RunIterative(ctx, ampest.IterativeConfig{
//     StatePrep: statePrep,
//     Oracle:    oracle,
//     NumQubits: ???,
//     MaxIters:  ???,
//     Shots:     1000,
// })
//
// TODO: Print and compare both results:
//   - Amplitude and error vs expected (0.5)
//   - Probability
//   - For iterative: confidence interval and iteration count
//   - Note the resource tradeoff: standard AE uses ancilla qubits, iterative does not
_ = context.Background()
fmt.Println("Uncomment the code above and fill in the config!")

Uncomment the code above and fill in the config!


### Exercise 3 -- Amplitude Estimation for a Biased State

Instead of the uniform superposition, prepare a **biased** state where qubit 0
has a rotation $R_y(\pi/3)$ and qubit 1 has $R_y(\pi/6)$. Mark the $|11\rangle$
state and use amplitude estimation to determine its probability.

*Hint:* The expected amplitude is $\sin(\pi/6) \cdot \sin(\pi/12) \approx 0.129$
and the expected probability is $\approx 0.0167$.

In [7]:
%%
// Exercise 3: Amplitude estimation with a biased state
//
// TODO: Build a state-prep circuit with Ry(pi/3) on qubit 0 and Ry(pi/6) on qubit 1
// statePrep, err := builder.New("biased", 2).Ry(0, math.Pi/3).Ry(1, math.Pi/6).Build()
//
// TODO: Create a phase oracle marking |11>
// oracle := grover.PhaseOracle([]int{???}, 2)
//
// TODO: Run amplitude estimation with 6 phase bits for higher precision
// result, err := ampest.Run(ctx, ampest.Config{
//     StatePrep:    statePrep,
//     Oracle:       oracle,
//     NumQubits:    ???,
//     NumPhaseBits: ???,
//     Shots:        1000,
// })
//
// TODO: Print estimated amplitude and probability, compare to expected values
//       Expected amplitude ≈ 0.129, expected probability ≈ 0.0167
_ = context.Background()
fmt.Println("Uncomment the code above and fill in the config!")

Uncomment the code above and fill in the config!


## Key Takeaways

1. **Quantum counting** uses QPE on the Grover iterate to estimate the
   number of solutions $M$ in a search space of size $N$. The Grover
   eigenvalue $e^{\pm 2i\theta}$ encodes $M/N = \sin^2(\theta)$.

2. **Amplitude estimation** generalizes quantum counting by estimating the
   probability amplitude of a "good" subspace prepared by an arbitrary
   state-preparation circuit. It is a core primitive for quantum speedups
   in Monte Carlo methods, optimization, and machine learning.

3. **Iterative amplitude estimation** achieves similar precision to standard
   AE without the large ancilla register, trading circuit width for depth
   through progressive Grover power doubling.

4. **Precision scales with phase bits**: more phase register qubits (or
   more iterations in the iterative variant) yield higher precision
   estimates at the cost of deeper circuits.

5. Both techniques provide a **quadratic speedup** over classical sampling,
   requiring $O(1/\epsilon)$ oracle calls to achieve precision $\epsilon$
   compared to the classical $O(1/\epsilon^2)$.

---

**Next up:** Notebook 11, where we will explore quantum error correction
and how to protect quantum information from noise.